# Imports

In [33]:
import json
import os
import shutil
import subprocess


from ase import Atoms
from ase.build import fcc100, molecule
from ase.calculators.emt import EMT
from ase.io import read, write
from ase.visualize import view

import numpy as np
import random

from symmetry_analysis.create import *
from symmetry_analysis.utility import *
from symmetry_analysis.transformations import *
from symmetry_analysis.symmetry import *
from symmetry_analysis.visualization import *
from symmetry_analysis.symmetry_with_sofi import *

from boss.bo.bo_main import BOMain
from boss.pp.pp_main import PPMain

In [34]:
def mol_to_symm(mol:Atoms):
    nat = len(mol)
    typ = np.array([s for s in mol.get_atomic_numbers()])
    coords = np.array([p for p in mol.get_positions()])
    
    sofi = ira_mod.SOFI()
    sym_thr = 0.05
    
    n_mat, mat_list = sofi.get_symm_ops(nat, typ, coords, sym_thr)
    rot_list = [x for x in mat_list if np.linalg.det(x)>0]

    # sym = sofi.compute(nat, typ, coords, sym_thr)
    # mat_list = sym.matrix
    # rot_list = [x for x in mat_list if np.linalg.det(x)>0]
    return rot_list

def get_symmetric_positions_and_angles_with_SOFI(
    pos: list, angles: list, slab: Atoms, mol: Atoms, idx=None, 
    x_bounds:list=[0., 1.], y_bounds:list=[0., 1.], chiral:bool=False, 
    output_filename:str='log.txt', tolerance:float=1e-3
):
    """
    Get unique equivalent positions in x, y, z and orientations (alpha, beta, gamma) based on the symmetries of a slab.
    Ensures unique pairs and that the z-coordinates match the original position's z within tolerance.
    Additionally logs symmetry operations to a file, including the determinant of each rotation.
    Parameters:
    pos (list): containing scaled coordinates x, y, z, respectively, of molecule center of mass position on slab.
    angles (list): containing alpha, beta, gamma angles, respectively, of molecule rotation.
    slab (ase.Atoms object): slab system isolated from the adsorbed molecule.
    mol (ase.Atoms object): molecule isolated from the slab. Also provides molecule symmetries via SOFI.
    idx (list): list of atom indeces in the molecule that will be used for plane reference in reflection cases.
    lower, upper_bound (int): Sets bounds for symmetric position search.
    chiral (bool): whether or not molecule is chiral
    output_filename (str): name of the file to write the symmetry operations and results.
    tolerance: tolerance for similarity in z position (in scaled coordinates).

    Returns:
    unique, duplicate, out_of_bounds: lists of unique, duplicate, and out-of-bounds adsorption configurations.
    """
    if len(x_bounds) != 2 or len(y_bounds) != 2:
        raise ValueError("Bounds for x or y must be specified as a list, i.e. [lowerbound, upperbound]")
        
    # if mol_symmetries is None:
    #     mol_symmetries = [np.eye(3)]
    
    if idx is None:
        idx = [0, 1, 2]
    if len(angles) != 3:
        raise ValueError("Angles must have exactly three elements (alpha, beta, gamma).")

    slab = slab.copy()
    mol = mol.copy()
    
    cell = get_spglib_cell(slab)
    symmetry = filter_z(get_symmetry(cell, 1e-5))
    dataset = get_symmetry_dataset(cell, 1e-5)
    P = dataset.transformation_matrix
    p = dataset.origin_shift
    pos = boss_to_symm(pos, slab)
    rotations = symmetry['rotations']
    translations = symmetry['translations']
    R_init = euler_to_rotation(angles[0], angles[1], angles[2], return_Rs=False)

    # integration of SOFI
    mol_symmetries = mol_to_symm(mol)
    
    unique_positions_and_angles = []
    duplicate_positions_and_angles = []
    out_of_bounds_points = []

    rotations_idx = index_arrays(rotations)
    translations_idx = index_arrays(translations)
    mol_symm_idx = index_arrays(mol_symmetries)
    
    with open(output_filename, 'w') as file:
        file.write("Symmetry Operations Check Log\n")
        file.write(f"Rotation matrices passed for operations are the following:\n")
        for i, rot in enumerate(unique_numpy_arrays(rotations)):
            file.write(f"{i+1}. \n{rot}\n")
        file.write(f"Translation vectors passed for operations are the following:\n")
        for i, trans in enumerate(unique_arrays(translations)):
            file.write(f"{i+1}. {trans}\n")
        file.write(f"Molecule symmetry matrices passed for operations are the following:\n")
        for i, symm in enumerate(unique_numpy_arrays(mol_symmetries)):
            file.write(f"{i+1}. \n{symm}\n")
        file.write("=" * 80 + "\n")
        file.write("Index | Pass/Fail | Failed Check | Position | Orientation | Det(W_surf) | W_surf | w_surf | R_symm | R_O'| UID\n")
        
        for i, (rotation, translation, rot_idx, trans_idx) in enumerate(zip(rotations, translations, rotations_idx, translations_idx)):
            determinant = np.linalg.det(rotation)
            if chiral and determinant < 0:
                file.write(f"{i+1} | Fail | Reflecting chiral molecule | {symm_to_boss(new_pos, slab)} | N/A | {determinant:.3f} | {rot_idx} | {trans_idx} | N/A | N/A | N/A\n")
                continue
            
            # new_pos = np.dot(rotation, pos) + translation
            new_pos = apply_position_transformation(pos, P, p, rotation, translation)
            new_pos = new_pos % 1.0
            new_pos_boss = symm_to_boss(new_pos, slab)

            out_of_bounds = False
            if not (x_bounds[0] <= new_pos_boss[0] <= x_bounds[1]) or not (y_bounds[0] <= new_pos_boss[1] <= y_bounds[1]):
                out_of_bounds = True

            # # if not (lb <= new_pos_boss[0] <= ub) or not (lb <= new_pos_boss[1] <= ub): CONDITION FOR X AND Y TO BE BETWEEN BOUNDS
            # if not (x_bounds[0] <= new_pos_boss[0] <= x_bounds[1]) or not (y_bounds[0] <= new_pos_boss[1] <= y_bounds[1]):
            #     uid = f"W{rot_idx}_w{trans_idx}_R0_O0"
            #     out_of_bounds_points.append(
            #         (
            #             np.array(new_pos_boss),
            #             np.array([]), # Empty array because O' not yet processed
            #             (rotation, translation),
            #             uid
            #         )
            #     )
            #     file.write(f"{i+1} | Fail | Out-of-bounds | {symm_to_boss(new_pos, slab)} | N/A | {determinant:.3f} | {rot_idx} | {trans_idx} | N/A | N/A | N/A\n")
            #     continue
            
            for (mol_symmetry, symm_idx) in zip(mol_symmetries, mol_symm_idx):
                rotation_trans = rotation_from_transformed(mol=mol, slab=slab, orientation=angles, rotation=rotation, translation=translation, idx=idx)
                R_new = np.dot(rotation_trans, np.dot(R_init, mol_symmetry))
                new_orientations = rotation_to_euler_rounded(R_new)

                for j, new_orientation in enumerate(new_orientations):
                    uid = f"{i+1}_W{rot_idx}_w{trans_idx}_R{symm_idx}_O{j+1}"
                    if out_of_bounds:
                        out_of_bounds_points.append(
                            (
                                np.array(new_pos_boss), 
                                np.array(new_orientation),
                                (rotation, translation),
                                uid
                            )
                        )
                        file.write(f"{i+1} | Fail | Out-of-bounds | {new_pos_boss} | {new_orientation} | {determinant:.3f} | {rot_idx} | {trans_idx} | {symm_idx} | {j+1} | {uid}\n")                    
                        continue
                        
                    pair_is_unique = True
                    for existing_pos, existing_angle, _, _ in unique_positions_and_angles:
                        if (np.allclose(new_pos_boss, existing_pos, atol=tolerance) and
                            np.allclose(new_orientation, existing_angle, atol=tolerance)):
                            pair_is_unique = False
                            break

                    if pair_is_unique:
                        unique_positions_and_angles.append(
                            (
                                np.array(new_pos_boss), 
                                np.array(new_orientation),
                                (rotation, translation),
                                uid
                            )
                        )
                        file.write(f"{i+1} | Pass | None | {new_pos_boss} | {new_orientation} | {determinant:.3f} | {rot_idx} | {trans_idx} | {symm_idx} | {j+1} | {uid}\n")
                    
                    else:
                        duplicate_positions_and_angles.append(
                            (
                                np.array(new_pos_boss), 
                                np.array(new_orientation),
                                (rotation, translation),
                                uid
                            )
                        )
                        file.write(f"{i+1} | Fail | Non-unique | {new_pos_boss} | {new_orientation} | {determinant:.3f} | {rot_idx} | {trans_idx} | {symm_idx} | {j+1} | {uid}\n")
    
    return unique_positions_and_angles, duplicate_positions_and_angles, out_of_bounds_points

# Obtaining symmetrically equivalent adsorption configurations

## Defining materials

In [35]:
wrkdir = os.getcwd()

In [36]:
au = fcc100('Au', size=(2, 2, 4), vacuum = 10)
water = molecule('H2O')

In [37]:
slab = au.copy()
mol = water.copy()

calculator = EMT()

slab.calc = calculator
mol.calc = calculator

e_slab = slab.get_potential_energy()
e_mol = mol.get_potential_energy()

In [38]:
idx = [0, 1, 2]

## Defining adsorption configurations

In [39]:
slab = au.copy()
mol = water.copy()

# initiating position variables
x = random.uniform(0,0.5)
y = random.uniform(0,0.5)
z = 2.8

# initiating orientation variables
a = random.randint(0, 360)
b = random.randint(0, 360)
c = random.randint(0, 360)

pos_list = [x, y, z]
angle_list = [a, b, c]

# initiating adsorption configuration
ads = create(slab, mol, a, b, c, x, y, z)
ads.calc = calculator

e_system = ads.get_potential_energy()
e_ads = e_system - e_slab - e_mol

print(f'The adsorption energy of the initial configuration is {e_ads}')

The adsorption energy of the initial configuration is 0.7284985038885248


In [40]:
unique, duplicate, out_of_bounds = get_symmetric_positions_and_angles_with_SOFI(
    pos_list,
    angle_list,
    slab,
    mol,
    idx,
    x_bounds=[0, 0.5],
    y_bounds=[0, 0.5],
)

In [41]:
unique_df = dataframe_from_s_prime(unique, slab, mol, ads)
print(unique_df[["Rotation Determinant", 'Adsorption Energy (eV)', 'Is it equivalent?']])

    Rotation Determinant  Adsorption Energy (eV)  Is it equivalent?
0                    1.0                  0.7285               True
1                    1.0                  0.7285               True
2                    1.0                  0.7285               True
3                    1.0                  0.7285               True
4                    1.0                  0.7285               True
5                    1.0                  0.7285               True
6                    1.0                  0.7285               True
7                    1.0                  0.7285               True
8                   -1.0                  0.7285               True
9                   -1.0                  0.7285               True
10                  -1.0                  0.7285               True
11                  -1.0                  0.7285               True
12                  -1.0                  0.7285               True
13                  -1.0                  0.7285

# Using in BOSS

In [42]:
def get_position_unique(data:list, n:int):
    # randomize order
    random.seed(42)
    random.shuffle(data)

    unique_positions = {}
    present_idx = []
    remains = []
    
    for i, s in enumerate(data):
        pos = tuple(s[0])
        if pos not in unique_positions:
            unique_positions[pos] = s
            present_idx.append(i)
    
    for i, s in enumerate(data):
        if i not in present_idx:
            remains.append(s)

    unique_s = list(unique_positions.values())

    if n <= len(unique_s):
        solutions = random.sample(unique_s, n)
        return(solutions)

    else:
        solutions = unique_s
        to_add = n-len(unique_s)
        if to_add < len(remains):
            solutions.extend(random.sample(remains, n-len(unique_s)))
        else:
            solutions.extend(remains)
        return(solutions)

In [43]:
au = fcc100('Au', size=(2, 2, 4), vacuum = 10)
water = molecule('H2O')

In [44]:
slab = au.copy()
mol = water.copy()
n_subsample = 4

bounds = np.array(
    [
        [0, 360],
        [0, 360],
        [0, 360],
        [0, 0.5],
        [0, 0.5],
        [2, 3]
    ]
)

kernels = np.array(
    [
        'stdp',
        'stdp',
        'stdp',
        'stdp',
        'stdp',
        'rbf'
    ]
)

In [45]:
calculator = EMT()

slab.calc = calculator
mol.calc = calculator
e_slab = slab.get_potential_energy()
e_mol = mol.get_potential_energy()

In [46]:
def func(x):
    x = np.squeeze(x)
    alpha, beta, gamma, a, b, c = x
    pos = [a, b, c]
    angle = [alpha, beta, gamma]

    ads = create(slab, mol, alpha, beta, gamma, a, b, c)
    ads.calc = calculator

    e_system = ads.get_potential_energy()
    e_ads = e_system - e_slab - e_mol

    if e_ads > 1:
        e_ads = np.log10(e_ads) + 1

    unique, _, _ = get_symmetric_positions_and_angles_with_SOFI(
        pos,
        angle,
        slab,
        mol,
        idx,
        x_bounds=[0, 0.5],
        y_bounds=[0, 0.5],
    )

    unique_subsampled = get_position_unique(unique, n_subsample)
    X_list = [np.array([*u[1], *u[0]]) for u in unique_subsampled]
    
    X = np.array(X_list)
    Y = np.ones(len(X_list)) * e_ads

    return X, Y

In [47]:
bo = BOMain(
    func,
    bounds,
    acqfn_name = 'exploit',
    kernel=kernels,
    initpts=10,
    iterpts=10
)

In [48]:
res = bo.run()

In [49]:
print('Predicted global min: ', res.select('mu_glmin', -1))

Predicted global min:  -0.2302830211260094
